# Reflexion Agent with Web Search

An agent that answers a question, critiques its own answer, searches the web to fill the gaps it found, and then revises the answer with citations. The loop is built with LangGraph.

Flow: `respond` -> `execute_tools` -> `revisor` -> (back to `execute_tools`, or END after `MAX_ITERATIONS` searches).

Set `OPENAI_API_KEY` and `TAVILY_API_KEY` as environment variables, or enter them when prompted.

In [ ]:
%pip install -q langchain-openai==0.3.10 langchain==0.3.21 openai==1.68.2 langchain-community==0.3.24 langgraph

In [ ]:
import os
import json
import getpass
from typing import List

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, MessageGraph


def _set_if_undefined(var: str) -> None:
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(var)


_set_if_undefined('OPENAI_API_KEY')
_set_if_undefined('TAVILY_API_KEY')

## Model, tool and prompt

In [ ]:
llm = ChatOpenAI(model='gpt-4.1-nano')
tavily_tool = TavilySearchResults(max_results=3)

prompt_template = ChatPromptTemplate.from_messages([
    (
        'system',
        '''You are an expert Incident Response Lead and Cyber Threat Intelligence Analyst. 
Your role is to analyze security alerts, investigate potential breaches, map adversary TTPs to MITRE ATT&CK, and evaluate technical evidence with forensic rigor.

Your response must follow these steps:
1. {first_instruction}
2. Identify potential threat actors, attack vectors, and MITRE ATT&CK techniques based on the observed activity.
3. Formulate technical hypotheses explaining how initial access, persistence, or execution occurred.
4. Reflect and critique your assessment: identify missing log artifacts, forensic ambiguities, alternative attack paths, or unverified assumptions.
5. After the reflection, **list 1-3 search queries separately** for researching specific CVE advisories, malware signatures, known C2 infrastructure, or threat actor campaigns. Do not include them inside the reflection.
'''
    ),
    MessagesPlaceholder(variable_name='messages'),
    (
        'system',
        "Analyze the security incident query above using the required format, focusing on technical depth and structured forensic triage."
    ),
])

## Structured output

The models below are bound to the LLM as tools so every response comes back in a fixed shape: answer, self-critique, and search queries (plus references when revising).

In [ ]:
class Reflection(BaseModel):
    missing: str = Field(description='What information is missing')
    superfluous: str = Field(description='What information is unnecessary')


class AnswerQuestion(BaseModel):
    answer: str = Field(description='Main response to the question')
    reflection: Reflection = Field(description='Self-critique of the answer')
    search_queries: List[str] = Field(description='Queries for additional research')


class ReviseAnswer(AnswerQuestion):
    '''Revise your original answer to your question.'''
    references: List[str] = Field(description='Citations motivating your updated answer.')

## Responder and revisor chains

In [ ]:
first_responder_prompt = prompt_template.partial(
    first_instruction='Provide a detailed ~250 word initial incident triage and threat assessment.'
)
initial_chain = first_responder_prompt | llm.bind_tools(tools=[AnswerQuestion])

revise_instructions = '''Revise your incident assessment using the new threat intelligence and search results, applying forensic rigor:
- Incorporate the critique to address unverified assumptions, missing telemetry, or alternative attack vectors.
- Correlate external threat intelligence (known malware behavior, C2 infrastructure, or CVE proof-of-concepts) with the internal symptoms.
- Map confirmed activity to specific MITRE ATT&CK IDs (e.g., T1059, T1021) and propose high-priority containment steps (host isolation, firewall blocklists, credential resets).
- Add a "References" section at the bottom citing relevant security advisories, vendor research, or CISA/MITRE alerts:
- [1] https://example.com/threat-report
- [2] https://example.com/cve-advisory
- Keep response under 250 words, emphasizing precision, containment speed, and evidence over speculation.
'''

revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)
revisor_chain = revisor_prompt | llm.bind_tools(tools=[ReviseAnswer])

## Tool execution and loop control

In [ ]:
MAX_ITERATIONS = 4


def execute_tools(state: List[BaseMessage]) -> List[BaseMessage]:
    '''Run the search queries requested in the last AI message.'''
    last_ai_message = state[-1]
    tool_messages = []
    for tool_call in last_ai_message.tool_calls:
        if tool_call['name'] in ['AnswerQuestion', 'ReviseAnswer']:
            search_queries = tool_call['args'].get('search_queries', [])
            query_results = {query: tavily_tool.invoke(query) for query in search_queries}
            tool_messages.append(
                ToolMessage(content=json.dumps(query_results), tool_call_id=tool_call['id'])
            )
    return tool_messages


def event_loop(state: List[BaseMessage]) -> str:
    '''Stop after MAX_ITERATIONS rounds of searching, otherwise search again.'''
    num_iterations = sum(isinstance(item, ToolMessage) for item in state)
    if num_iterations >= MAX_ITERATIONS:
        return END
    return 'execute_tools'

## Graph

In [ ]:
graph = MessageGraph()

graph.add_node('respond', initial_chain)
graph.add_node('execute_tools', execute_tools)
graph.add_node('revisor', revisor_chain)

graph.add_edge('respond', 'execute_tools')
graph.add_edge('execute_tools', 'revisor')
graph.add_conditional_edges('revisor', event_loop)
graph.set_entry_point('respond')

app = graph.compile()

## Run

In [ ]:
responses = app.invoke(
    '''We detected unauthorized PowerShell execution spawning rundll32.exe on our domain controller, 
    connecting out to an IP associated with Cobalt Strike beacons. What attack technique are we looking at, 
    how did they likely pivot, and what should our immediate containment actions be?'''
)

In [ ]:
# Collect every draft the agent produced, in order
answers = []
for msg in responses:
    for tool_call in getattr(msg, 'tool_calls', None) or []:
        answer = tool_call.get('args', {}).get('answer')
        if answer:
            answers.append(answer)

print('--- Initial draft ---')
print(answers[0])

for i, ans in enumerate(answers[1:-1], start=1):
    print(f'\n--- Revision {i} ---')
    print(ans)

print('\n--- Final answer ---')
print(answers[-1])